<a href="https://colab.research.google.com/github/shauna-rsh/abalone-age/blob/main/abalone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [74]:
!wget -O abalone.csv https://raw.githubusercontent.com/shauna-rsh/abalone-age/refs/heads/main/abalone.csv?token=GHSAT0AAAAAADW6SP52B5WW4PBBF5BNMHYC2NN2OUQ

--2026-03-08 20:01:30--  https://raw.githubusercontent.com/shauna-rsh/abalone-age/refs/heads/main/abalone.csv?token=GHSAT0AAAAAADW6SP52B5WW4PBBF5BNMHYC2NN2OUQ
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-08 20:01:30 ERROR 404: Not Found.



In [69]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn import preprocessing, svm
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
!pip install shap
import shap


In [70]:
columns = ["Sex", "Length", "Diameter", "Height",
    "Whole_weight", "Shucked_weight",
    "Viscera_weight", "Shell_weight", "Rings"]

df = pd.read_csv(
    "abalone.csv",
    header=None,
    names=columns
)

df.head()

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings


One hot encoding of Sex category. Checking if there are any invalid rows by seeing if a cell is null.Identification of outliers via z score.

In [71]:
df = pd.get_dummies(df, columns=["Sex"], drop_first=True) #one-hot encoding

In [72]:
df.isnull().sum()

,0
Length,0
Diameter,0
Height,0
Whole_weight,0
Shucked_weight,0
Viscera_weight,0
Shell_weight,0
Rings,0


In [73]:
def outliers_z_score(data, columns):
  threshold = 2
  all_outlier_indices = set()
  for c in columns:
    z = np.abs(stats.zscore(data[c])) # Use 'data' instead of 'df' to keep consistent if original 'data' was modified before this function call.
    outlier_indices = np.where(z > threshold)[0]
    all_outlier_indices.update(outlier_indices)
  data = data.drop(list(all_outlier_indices))
  return data
print("DF shape with outliers:",df.shape)
df = outliers_z_score(df, df.columns)
print("DF shape without outliers:",df.shape)

DF shape with outliers: (0, 8)


AttributeError: 'numpy.dtypes.ObjectDType' object has no attribute 'dtype'

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=False, cmap="magma")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
correlation_table = df.corr()
print(correlation_table)

Highest correlation seems to be with shucked weight (weight of the abalone meat only).


In [ ]:
df['Age']=df['Rings']+1.5
df.head()

In [ ]:
df.plot()
plt.show()

Relationship between age and length and weight.

In [ ]:
ax1= df.plot(kind='scatter', x='Age', y='Length', label='length')
df.plot.scatter(x='Age', y='Diameter', color='Red', label='diameter', ax=ax1)
df.plot.scatter(x='Age', y='Height', color='Green', label='height', ax=ax1)
plt.show()

In [ ]:
ax2 = df.plot(kind='scatter', x='Age', y='Whole_weight', color='DarkBlue', label='whole weight data')
df.plot.scatter(x='Age', y='Shucked_weight', color='Red', label='shucked weight data', ax=ax2)
df.plot.scatter(x='Age', y='Viscera_weight', color='Green', label='viscera weight data', ax=ax2)
df.plot.scatter(x='Age', y='Shell_weight', color='Yellow', label='shell weight data', ax=ax2)
plt.show()

Pre-processing for the models

In [ ]:
X = df.drop(["Rings", "Age"], axis=1)
y = df["Age"]
#divide data into training and testing sets to evaluate generalization performance. 80 / 20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler() #standardizes features by removing the mean and scaling them to unit variance.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
def model_evaluation(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    return r2, mae, rmse

Linear Regression w/ Pred vs actual Scatterplot & Eval

In [ ]:
linregr = LinearRegression()
linregr_result = model_evaluation(
 linregr,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)
linregr.fit(X_train_scaled, y_train)
y_pred = linregr.predict(X_test_scaled)

In [ ]:
explainer = shap.LinearExplainer(linregr, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)
shap.summary_plot(
    shap_values,
    X_test_scaled,
    feature_names=X.columns
)

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Actual Age")
plt.ylabel("Predicted Age")
plt.title("Linear Regression Predictions")
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color='red')
plt.show()

Ridge Regression  w/ Pred vs actual Scatterplot & Eval

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
pred_basic = ridge.predict(X_test)
ridge_result = model_evaluation(
    ridge,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

In [ ]:
explainer = shap.LinearExplainer(ridge, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)
shap.summary_plot(
    shap_values,
    X_test_scaled,
    feature_names=X.columns
)

In [ ]:
plt.scatter(y_test, pred_basic, alpha=0.6)
plt.xlabel("Actual Age")
plt.ylabel("Predicted Age")
plt.title("Ridge Regression Predictions")
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color='red')
plt.show()

Random Forest Regressor w/ SHAP and Eval

In [ ]:
forestregr = RandomForestRegressor(n_estimators=200, random_state=42, oob_score= True)
forestregr.fit(X_train, y_train)
y_pred = forestregr.predict(X_test)
forestregr_result = model_evaluation(
    forestregr,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Actual Age")
plt.ylabel("Predicted Age")
plt.title("Predicted vs Actual Abalone Age")
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color="red")
plt.show()

In [ ]:
explainer = shap.TreeExplainer(regressor)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

Gradient Boosting

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)
gbr.fit(X_train, y_train)
pred_gb = gbr.predict(X_test)
gbr_result = model_evaluation(
    gbr,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

scores = cross_val_score(gbr, X_train, y_train, cv=5, scoring='r2')
print("Cross-validation R² scores:", scores)
print("Mean R² score:", scores.mean())

In [ ]:
explainer = shap.TreeExplainer(gbr)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

In [ ]:
plt.scatter(y_test, pred_gb, alpha=0.6)
plt.xlabel("Actual Age")
plt.ylabel("Predicted Age")
plt.title("Predicted vs Actual Abalone Age")
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color="red")
plt.show()

Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression", "Random Forest", "Gradient Boosting"],
    "R² Score": [linregr_result[0], ridge_result[0], forestregr_result[0], gbr_result[0]],
    "MAE": [linregr_result[1], ridge_result[1], forestregr_result[1], gbr_result[1]],
    "RMSE": [linregr_result[2], ridge_result[2], forestregr_result[2], gbr_result[2]]
})
results.sort_values(by="R² Score", ascending=False)
results.set_index("Model")["R² Score"].plot(kind="bar")

Best model with the highest R² Score is GBR


In [ ]:
gbr.feature_importances_
importance = pd.Series(gbr.feature_importances_, index=X_train.columns)
importance.sort_values().plot(kind="barh")

In [ ]:
residuals = y_test - pred_gb
plt.scatter(pred_gb, residuals)
plt.axhline(0)
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.show()

In [ ]:
errors = np.abs(y_test - pred_gb)

plt.scatter(y_test, pred_gb, c=errors)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
)

plt.xlabel("Actual Age")
plt.ylabel("Predicted Age")
plt.title("Prediction Error Visualization")
plt.colorbar(label="Absolute Error")
plt.show()